In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.models import resnet50, ResNet50_Weights
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np

# Force use of GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)


Using device: cuda


In [2]:

# Updated weights API, using default pretrained weights
weights = ResNet50_Weights.DEFAULT

# Define transforms with normalization matching the weights' pretrained stats
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((300, 225)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        weights.transforms().normalize  # Use built-in normalization for these weights
    ]),
    'test': transforms.Compose([
        transforms.Resize((300, 225)),
        transforms.ToTensor(),
        weights.transforms().normalize
    ])
}

# Data directory - change if needed, assume dataset is organized as before
data_dir = './new_data_balanced'

# Load datasets
image_datasets = {
    x: datasets.ImageFolder(root=f"{data_dir}/{x}", transform=data_transforms[x])
    for x in ['train', 'test']
}

# Data loaders
batch_size = 32  # Increased batch size for better GPU utilization
dataloaders = {
    x: DataLoader(image_datasets[x], batch_size=batch_size, shuffle=(x=='train'), num_workers=4, pin_memory=True)
    for x in ['train', 'test']
}

dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'test']}
class_names = image_datasets['train'].classes
print(f"Classes: {class_names}")
print(f"Dataset sizes: {dataset_sizes}")


AttributeError: 'ImageClassification' object has no attribute 'normalize'

In [ ]:

# Load model with pretrained weights
model = resnet50(weights=weights)

# Replace the final fully connected layer for binary classification
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)

model = model.to(device)

# Define loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

# Enable cudnn benchmark for potential speedup on fixed size inputs
torch.backends.cudnn.benchmark = True

# Training loop
num_epochs = 10
best_acc = 0.0


In [ ]:

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print('-' * 20)
    
    for phase in ['train', 'test']:
        if phase == 'train':
            model.train()
        else:
            model.eval()
        
        running_loss = 0.0
        running_corrects = 0
        
        for inputs, labels in dataloaders[phase]:
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            optimizer.zero_grad()
            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                _, preds = torch.max(outputs, 1)
                
                if phase == 'train':
                    loss.backward()
                    optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)
        
        epoch_loss = running_loss / dataset_sizes[phase]
        epoch_acc = running_corrects.double().item() / dataset_sizes[phase]
        
        print(f"{phase.capitalize()} Loss: {epoch_loss:.4f}  Acc: {epoch_acc*100:.2f}%")
        
        if phase == 'test' and epoch_acc > best_acc:
            best_acc = epoch_acc
            torch.save(model.state_dict(), 'best_model.pth')
            print(f"Model saved with accuracy: {best_acc*100:.2f}%")


In [ ]:

# Load best model for evaluation
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

# Gather predictions and labels for confusion matrix
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in dataloaders['test']:
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Confusion matrix
conf_matrix = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()
